In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv("data/heart_disease_uci.csv")

df.drop(columns=["id", "dataset", "thal", "slope"], inplace=True)

df["num"] = df["num"].apply(lambda x: 1 if x > 0 else 0)

df["chol"] = df["chol"].replace(0, np.nan)

In [ ]:
df["sex"] = df["sex"].astype(str).str.lower().map({"male": 1, "female": 0})
df["fbs"] = df["fbs"].astype(str).str.lower().map({"true": 1, "false": 0})
df["exang"] = df["exang"].astype(str).str.lower().map({"true": 1, "false": 0})

display(df.head())

In [ ]:
X = df.drop(columns=["num"])
y = df["num"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

In [ ]:
num_cols = ["age", "trestbps", "chol", "thalch", "oldpeak", "ca"]
cat_cols = ["cp", "restecg"]
bin_cols = ["sex", "fbs", "exang"]

num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) 
])

bin_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols),
        ('bin', bin_pipeline, bin_cols)
    ],
    remainder='drop' 
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

cat_encoder = preprocessor.named_transformers_['cat'].named_steps['encoder']
one_hot_cols = cat_encoder.get_feature_names_out(cat_cols)
final_columns = num_cols + list(one_hot_cols) + bin_cols

print("Pipeline completed successfully.")
print(f"Final training data shape: {X_train_processed.shape}")
print(f"Total features: {len(final_columns)}")

In [ ]:
X_train_df = pd.DataFrame(X_train_processed, columns=final_columns)

print("Processed Training Data:")
display(X_train_df.head())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

classifier = LogisticRegression(max_iter=1000, random_state=42)

classifier.fit(X_train_processed, y_train)

y_pred_log = classifier.predict(X_test_processed)

print("--- Logistic Regression Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_log):.4f}\n")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_log))